In [ ]:
import mlflow
import mlflow.spark
from pyspark.sql.functions import col, when, hour, dayofweek, month
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# Set MLflow experiment
mlflow.set_experiment("/Banking-Fraud-Detection")

print("🔹 Starting ML Fraud Detection Training")

In [ ]:
# Load training data from Silver layer
SOURCE_TABLE = "banking.banking.banking_silver_transactions"

df = spark.table(SOURCE_TABLE)

# Filter for records with fraud labels (assuming historical data has labels)
training_df = df.filter(col("is_fraud").isNotNull())

print(f"Training records: {training_df.count()}")

In [ ]:
# Feature Engineering
training_df = training_df.withColumn("hour", hour(col("timestamp"))) \
                         .withColumn("day_of_week", dayofweek(col("timestamp"))) \
                         .withColumn("month", month(col("timestamp")))

# Index and encode categorical features
indexers = [
    StringIndexer(inputCol="transaction_type", outputCol="transaction_type_index"),
    StringIndexer(inputCol="location", outputCol="location_index")
]

encoders = [
    OneHotEncoder(inputCol="transaction_type_index", outputCol="transaction_type_vec"),
    OneHotEncoder(inputCol="location_index", outputCol="location_vec")
]

# Assemble features
assembler = VectorAssembler(
    inputCols=["amount", "transaction_type_vec", "location_vec", "hour", "day_of_week", "month"],
    outputCol="features"
)

# Model
rf = RandomForestClassifier(labelCol="is_fraud", featuresCol="features", numTrees=100)

# Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [ ]:
# Split data
train_df, test_df = training_df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count()}, Test: {test_df.count()}")

In [ ]:
# Train model with MLflow tracking
with mlflow.start_run():
    model = pipeline.fit(train_df)
    
    # Evaluate
    predictions = model.transform(test_df)
    evaluator = BinaryClassificationEvaluator(labelCol="is_fraud", metricName="areaUnderROC")
    auc = evaluator.evaluate(predictions)
    
    # Log metrics
    mlflow.log_metric("auc", auc)
    mlflow.log_param("num_trees", 100)
    
    # Log model
    mlflow.spark.log_model(model, "fraud-detection-model")
    
    print(f"Model trained with AUC: {auc}")

In [ ]:
# Register model in MLflow Model Registry
client = mlflow.tracking.MlflowClient()
model_version = mlflow.register_model(
    f"runs:/{mlflow.active_run().info.run_id}/fraud-detection-model",
    "FraudDetectionModel"
)

print(f"Model registered: {model_version.version}")